# Semantic Kernel 

ਇਸ ਕੋਡ ਨਮੂਨੇ ਵਿੱਚ, ਤੁਸੀਂ ਇੱਕ ਸਧਾਰਣ ਏਜੰਟ ਬਣਾਉਣ ਲਈ [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI ਫਰੇਮਵਰਕ ਦੀ ਵਰਤੋਂ ਕਰੋਗੇ।

ਇਸ ਨਮੂਨੇ ਦਾ ਮਕਸਦ ਉਹ ਕਦਮ ਦਰਸਾਉਣਾ ਹੈ ਜੋ ਬਾਅਦ ਵਿੱਚ ਵੱਖ-ਵੱਖ ਏਜੰਟਿਕ ਪੈਟਰਨ ਲਾਗੂ ਕਰਨ ਵਾਲੇ ਹੋਰ ਕੋਡ ਨਮੂਨਿਆਂ ਵਿੱਚ ਅਮਲ ਕੀਤੇ ਜਾਣਗੇ।


## ਲੋੜੀਏ ਪਾਈਥਨ ਪੈਕੇਜਜ਼ ਨੂੰ ਇੰਪੋਰਟ ਕਰੋ


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## ਕਲਾਇੰਟ ਬਣਾਉਣਾ

ਇਸ ਉਦਾਹਰਨ ਵਿਚ, ਅਸੀਂ LLM ਤੱਕ ਪਹੁੰਚ ਕਰਨ ਲਈ [GitHub ਮਾਡਲਾਂ](https://aka.ms/ai-agents-beginners/github-models) ਦੀ ਵਰਤੋਂ ਕਰਾਂਗੇ।

`ai_model_id` ਨੂੰ `gpt-4o-mini` 'ਤੇ ਸੈੱਟ ਕੀਤਾ ਗਿਆ ਹੈ। ਵੱਖ-ਵੱਖ ਨਤੀਜੇ ਵੇਖਣ ਲਈ ਮਾਡਲ ਨੂੰ GitHub ਮਾਡਲ ਮਾਰਕੀਟਪਲੇਸ ਵਿੱਚ ਉਪਲਬਧ ਕਿਸੇ ਹੋਰ ਮਾਡਲ ਨਾਲ ਬਦਲ ਕੇ ਦੇਖੋ।

`base_url` ਲਈ GitHub ਮਾਡਲਾਂ ਲਈ ਜਰੂਰੀ `Azure Inference SDK` ਦੀ ਵਰਤੋਂ ਕਰਨ ਲਈ, ਅਸੀਂ Semantic Kernel ਵਿੱਚ `OpenAIChatCompletion` ਕਨੈਕਟਰ ਦੀ ਵਰਤੋਂ ਕਰਾਂਗੇ। ਹੋਰ ਵੀ [ਉਪਲਬਧ ਕਨੈਕਟਰ](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) ਹਨ ਜੋ ਤੁਹਾਨੂੰ Semantic Kernel ਨੂੰ ਹੋਰ ਮਾਡਲ ਪ੍ਰਦਾਤਿਆਂ ਨਾਲ ਵਰਤਣ ਦੀ ਆਗਿਆ ਦਿੰਦੇ ਹਨ।


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.github.ai/inference",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## Creating the Agent

ਇਥੇ, ਅਸੀਂ `TravelAgent` ਨਾਮਕ ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ।

ਇਸ ਉਦਾਹਰਣ ਵਿੱਚ, ਅਸੀਂ ਬਹੁਤ ਹੀ ਮੁਢਲੇ ਹੁਕਮਾਂ ਦਾ ਇਸਤੇਮਾਲ ਕਰਦੇ ਹਾਂ। ਜਿਸ ਤਰ੍ਹਾਂ ਏਜੰਟ ਦਾ ਵਿਹਾਰ ਬਦਲਦਾ ਹੈ, ਇਸਨੂੰ ਵੇਖਣ ਲਈ ਇਹ ਹੁਕਮ ਤੁਹਾਡੇ ਮਨਮੁਤਾਬਕ ਬਦਲ ਸਕਦੇ ਹੋ।


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## Running the Agents

ਹੁਣ ਅਸੀਂ ਏਜੰਟ ਨੂੰ ਚਲਾਉਣ ਲਈ `ChatHistory` ਸੈੱਟ ਕਰਦੇ ਹਾਂ ਅਤੇ ਇਸ ਵਿੱਚ `system_message` ਨੂੰ ਸ਼ਾਮਲ ਕਰਦੇ ਹਾਂ। ਅਸੀਂ ਪਹਿਲਾਂ ਜੋ `AGENT_INSTRUCTIONS` ਬਣਾਈਆਂ ਸੀ, ਉਹ ਵਰਤਾਂਗੇ।

ਇਹ ਸੈਟ ਹੋਣ ਦੇ ਬਾਅਦ, ਅਸੀਂ `user_inputs` ਬਣਾਉਂਦੇ ਹਾਂ, ਜੋ ਦੱਸਦਾ ਹੈ ਕਿ ਯੂਜ਼ਰ ਏਜੰਟ ਨੂੰ ਕੀ ਭੇਜ ਰਿਹਾ ਹੈ। ਇਸ ਉਦਾਹਰਣ ਵਿੱਚ, ਸੁਨੇਹਾ `Plan me a sunny vacation` ਸੈੱਟ ਕੀਤਾ ਗਿਆ ਹੈ।

ਤੁਸੀਂ ਇਸ ਸੁਨੇਹੇ ਨੂੰ ਬਦਲ ਸਕਦੇ ਹੋ ਦੇਖਣ ਲਈ ਕਿ ਏਜੰਟ ਕਿਵੇਂ ਵੱਖਰੇ ਤਰੀਕੇ ਨਾਲ ਪ੍ਰਤੀਕਿਰਿਆ ਕਰਦਾ ਹੈ।


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
